# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published on: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `@id` field of each entity in the dataset ensures unique identification. We'll list all Record Sets and their fields with `@id` so you can reference any data element precisely.

In [ ]:
# List all record sets and their field @id
from pprint import pprint

print('Available Record Sets:')
record_sets = metadata.recordSets
for rset in record_sets:
    print(f"  - Name: {rset.name}")
    print(f"    @id: {rset.id}")
    print(f"    Description: {rset.description}")
    print("    Fields:")
    for field in rset.fields:
        print(f"      - {field.name} (@id: {field.id}) Data type: {field.dataType}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.


In [ ]:
# Choose record sets by @id
# Copy one or more @id values from the previous cell's output.

# Example: Suppose one record set is '@id: ms:main-proteins'
record_sets_ids = [
    # Replace with the actual record set IDs found above
    'ms:main-proteins',  # Example placeholder
]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Extracting records from: {record_set_id}")
    # List of dicts, one record per row
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records. Fields: {list(df.columns)}\n")
# Review the first few rows of the first record set
if record_sets_ids:
    display(dataframes[record_sets_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate on one record set using field `@id`s. Replace `protein_abundance` and `sample_type` with the exact field `@id`s from your dataset (see Data Overview above).

In [ ]:
# Select record set and fields by @id
record_set_id = record_sets_ids[0]  # Update if needed

# Example field IDs (replace with actual @id from Data Overview)
numeric_field_id = 'protein_abundance'   # e.g. 'protein_abundance', use exact @id
group_field_id = 'sample_type'           # e.g. 'sample_type', use exact @id
threshold = 10

df = dataframes[record_set_id]

# Check that field exists
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()

    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped mean by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Field {group_field_id} not present in this record set.")
else:
    print(f"Field {numeric_field_id} not found in columns: {list(df.columns)}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].astype(float), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset, loaded via Croissant, provides fields for protein abundance and sample type, among others.
- Filtering, normalization, and grouping enable in-depth analysis of mass spectrometry results.
- Visualizations confirm the data's structure and distribution.

**Replace placeholder field @id and record set @id with actual values from your dataset for your specific analysis.**